In [ ]:
import warnings
warnings.filterwarnings('ignore')

# RNA velocity: data preparation

In [ ]:
import scanpy as sc
import anndata
from scipy import io
from scipy.sparse import coo_matrix, csr_matrix
import numpy as np
import os
import pandas as pd

# load sparse matrix:
X = io.mmread("../files_JULY_2023/counts.mtx")

# create anndata object
adata = anndata.AnnData(
    X=X.transpose().tocsr()
)

# load cell metadata:
cell_meta = pd.read_csv("../files_JULY_2023/metadata.csv")
cell_meta["DM_1"]= -cell_meta["DM_1"]

# load gene names:
with open("../files_JULY_2023/gene_names.csv", 'r') as f:
    gene_names = f.read().splitlines()

# set anndata observations and index obs by barcodes, var by gene names
adata.obs = cell_meta
adata.obs.index = adata.obs['barcode']
adata.var.index = gene_names

# load dimensional reduction:
pca = pd.read_csv("../files_JULY_2023/pca.csv")
pca.index = adata.obs.index

# set pca and umap
adata.obsm['X_pca'] = pca.to_numpy()
adata.obsm['X_umap'] = np.vstack((adata.obs['DM_1'].to_numpy(), adata.obs['DM_2'].to_numpy())).T

In [ ]:
adata.uns["integrated_snn_res.0.3_colors"]=["#A58AFF","#F8766D", "#C49A00", "#53B400", "#00B6EB", "#FB61D7"]

In [ ]:
adata.obs['integrated_snn_res.0.1'] = adata.obs['integrated_snn_res.0.1'].astype('category')
adata.obs['integrated_snn_res.0.2'] = adata.obs['integrated_snn_res.0.2'].astype('category')
adata.obs['integrated_snn_res.0.3'] = adata.obs['integrated_snn_res.0.3'].astype('category')

# save dataset as anndata format
adata.write('../files_JULY_2023/my_data.h5ad')

In [ ]:
# plot a UMAP colored by sampleID to test:
sc.pl.umap(adata, frameon=True,color=['integrated_snn_res.0.1','integrated_snn_res.0.2',
                                         'integrated_snn_res.0.3','cond'], save=False)

In [ ]:
import scvelo as scv
import scanpy as sc
import cellrank as cr
import numpy as np
import pandas as pd
import anndata as ad

In [ ]:
scv.settings.verbosity = 3
scv.settings.set_figure_params('scvelo', facecolor='white', dpi=100, frameon=False)
cr.settings.verbosity = 2

In [ ]:
adata = scv.read('../files_JULY_2023/my_data.h5ad')

In [ ]:
adata

In [ ]:
adataWT=adata[adata.obs.cond=='WT']
adatamut=adata[adata.obs.cond=='mutant']

In [ ]:
# load loom files for spliced/unspliced matrices for each sample:
ldataWT = scv.read('../../../RNA_velo_data/WT/adata.h5ad', cache=True)
ldatamut = scv.read('../../../RNA_velo_data/mutant/adata.h5ad', cache=True)

In [ ]:
ldataWT.var_names

In [ ]:
ldataWT.var.index=ldataWT.var.gene_name.astype(str)
ldatamut.var.index=ldatamut.var.gene_name.astype(str)

In [ ]:
intersWT=list(set(adataWT.var_names).intersection(ldataWT.var_names))
intersmut=list(set(adatamut.var_names).intersection(ldatamut.var_names))

In [ ]:
# make variable names unique
ldataWT.var_names_make_unique()
ldatamut.var_names_make_unique()

# make variable names unique
adataWT.var_names_make_unique()
adatamut.var_names_make_unique()

adataWT=adataWT[:,intersWT]
ldataWT=ldataWT[:,intersWT]
adatamut=adatamut[:,intersmut]
ldatamut=ldatamut[:,intersmut]

In [ ]:
# rename barcodes in order to merge:
barcodes = [bc.split('-')[0] for bc in adataWT.obs.index.tolist()]
#barcodes = [bc[0:len(bc)-1] + '_10' for bc in barcodes]
adataWT.obs.index = barcodes

barcodes = [bc.split('-')[0] for bc in adatamut.obs.index.tolist()]
#barcodes = [bc[0:len(bc)-1] + '_11' for bc in barcodes]
adatamut.obs.index = barcodes

In [ ]:
# concatenate the three loom
#ldata = ldata1.concatenate([ldata2, ldata3])
# merge matrices into the original adata object
adataWT = scv.utils.merge(ldataWT,adataWT,copy=True)
adatamut = scv.utils.merge(ldatamut,adatamut,copy=True)
# plot umap to check
sc.pl.umap(adataWT, color='cond', frameon=False, legend_loc='on data')#, title='', save='_celltypes.pdf')
sc.pl.umap(adatamut, color='cond', frameon=False, legend_loc='on data')#, title='', save='_celltypes.pdf')

In [ ]:
adata.obs.cond.value_counts()

In [ ]:
adatamut

In [ ]:
scv.pl.proportions(adataWT, groupby='integrated_snn_res.0.1')

In [ ]:
scv.pl.proportions(adatamut, groupby='integrated_snn_res.0.1')

In [ ]:
adataWT[:,"Xist"].layers["spliced"].todense().sum()

In [ ]:
adataWT[:,"Xist"].layers["unspliced"].todense().sum()

In [ ]:
print(adataWT[:,"Mecp2"].layers["spliced"].todense().sum())
print(adataWT[:,"Mecp2"].layers["unspliced"].todense().sum())

In [ ]:
print(adataWT[:,"Cdkl5"].layers["spliced"].todense().sum())
print(adataWT[:,"Cdkl5"].layers["unspliced"].todense().sum())

In [ ]:
scv.pl.scatter(adataWT, ["Mecp2","Cdkl5"], frameon=True, color='integrated_snn_res.0.3', size=10, linewidth=1.5)



# WT

In [ ]:
scv.pp.filter_genes(adataWT,min_cells=10)
scv.pp.filter_and_normalize(adataWT, min_counts=20, min_counts_u=10, n_top_genes=2000)
adataWT.raw=adataWT
scv.pp.moments(adataWT, n_pcs=30, n_neighbors=30)
scv.tl.recover_dynamics(adataWT,n_jobs=8)
scv.tl.velocity(adataWT, mode='dynamical')
scv.tl.velocity_graph(adataWT)
scv.tl.velocity_embedding(adataWT, basis='umap') 
scv.tl.paga(adataWT, groups='integrated_snn_res.0.3')
df = scv.get_df(adataWT, 'paga/transitions_confidence', precision=2).T
print('velo genes',adataWT.var['velocity_genes'].sum())

In [ ]:
scv.pl.velocity_embedding_stream(adataWT,basis='umap', legend_loc='on data',dpi=100,
                                 color='integrated_snn_res.0.1',
                                     size=100,alpha=1,title='WT',
                                     save='RNAvelo_res01_WT.pdf')
scv.pl.velocity_embedding_stream(adataWT,basis='umap', legend_loc='on data',dpi=100,
                                 color='integrated_snn_res.0.2',
                                     size=100,alpha=1,title='WT',
                                     save='RNAvelo_res02_WT.pdf')
scv.pl.velocity_embedding_stream(adataWT,basis='umap', legend_loc='on data',dpi=100,
                                 color='integrated_snn_res.0.3',
                                     size=100,alpha=1,title='WT',
                                     save='RNAvelo_res03_WT.pdf')

In [ ]:
scv.tl.rank_velocity_genes(adataWT, groupby='integrated_snn_res.0.3', min_corr=.3)

df = scv.DataFrame(adataWT.uns['rank_velocity_genes']['names'])
df.head()

In [ ]:
scv.pl.scatter(adataWT, df['0'][:3], ylabel='cl0', frameon=False, color='integrated_snn_res.0.3', size=10, linewidth=1.5)
scv.pl.scatter(adataWT, df['1'][:3], ylabel='cl1', frameon=False, color='integrated_snn_res.0.3', size=10, linewidth=1.5)
scv.pl.scatter(adataWT, df['2'][:3], ylabel='cl2', frameon=False, color='integrated_snn_res.0.3', size=10, linewidth=1.5)

In [ ]:
scv.tl.velocity_confidence(adataWT)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adataWT, c=keys, cmap='coolwarm', perc=[5, 95])

In [ ]:
# scv.tl.latent_time(adataWT)
# scv.pl.scatter(adataWT, color='latent_time', color_map='gnuplot', size=80)

In [ ]:
# top_genes = adataWT.var['fit_likelihood'].sort_values(ascending=False).index[:300]
# scv.pl.heatmap(adataWT, var_names=top_genes, sortby='latent_time', col_color='integrated_snn_res.0.1', n_convolve=100)

In [ ]:
# var_names = [ 'Nanog', 'Dnmt3b']
# scv.pl.scatter(adataWT, var_names, color='integrated_snn_res.0.1', frameon=False)
# scv.pl.scatter(adataWT, x='latent_time', y=var_names, color='integrated_snn_res.0.1', frameon=False)

# MUTANT

In [ ]:
scv.pp.filter_genes(adatamut,min_cells=10)
scv.pp.filter_and_normalize(adatamut, min_counts=20, min_counts_u=10, n_top_genes=2000)
adatamut.raw=adatamut
scv.pp.moments(adatamut, n_pcs=30, n_neighbors=30)
scv.tl.recover_dynamics(adatamut,n_jobs=8)
scv.tl.velocity(adatamut, mode='dynamical')
scv.tl.velocity_graph(adatamut)
scv.tl.velocity_embedding(adatamut, basis='umap') 
scv.tl.paga(adatamut, groups='integrated_snn_res.0.3')
df = scv.get_df(adatamut, 'paga/transitions_confidence', precision=2).T
print('velo genes',adatamut.var['velocity_genes'].sum())

In [ ]:
scv.pl.velocity_embedding_stream(adatamut,basis='umap', legend_loc='on data',dpi=100,
                                 color='integrated_snn_res.0.1',
                                     size=100,alpha=1,title='mutant',
                                     save='RNAvelo_res01_mut.png')
scv.pl.velocity_embedding_stream(adatamut,basis='umap', legend_loc='on data',dpi=100,
                                 color='integrated_snn_res.0.2',
                                     size=100,alpha=1,title='mutant',
                                     save='RNAvelo_res02_mut.png')
scv.pl.velocity_embedding_stream(adatamut,basis='umap', legend_loc='on data',dpi=100,
                                 color='integrated_snn_res.0.3',
                                     size=100,alpha=1,title='mutant',
                                     save='RNAvelo_res03_mut.png')

In [ ]:
scv.tl.rank_velocity_genes(adatamut, groupby='integrated_snn_res.0.3', min_corr=.3)

df = scv.DataFrame(adatamut.uns['rank_velocity_genes']['names'])
df.head()

In [ ]:
scv.pl.scatter(adatamut, df['0'][:3], ylabel='cl0', frameon=False, color='integrated_snn_res.0.3', size=10, linewidth=1.5)
scv.pl.scatter(adatamut, df['1'][:3], ylabel='cl1', frameon=False, color='integrated_snn_res.0.3', size=10, linewidth=1.5)
scv.pl.scatter(adatamut, df['2'][:3], ylabel='cl2', frameon=False, color='integrated_snn_res.0.3', size=10, linewidth=1.5)

In [ ]:
scv.tl.velocity_confidence(adatamut)
keys = 'velocity_length', 'velocity_confidence'
scv.pl.scatter(adatamut, c=keys, cmap='coolwarm', perc=[5, 95])

In [ ]:
# scv.tl.latent_time(adatamut)
# scv.pl.scatter(adatamut, color='latent_time', color_map='gnuplot', size=80)

In [ ]:
# top_genes = adatamut.var['fit_likelihood'].sort_values(ascending=False).index[:300]
# scv.pl.heatmap(adatamut, var_names=top_genes, sortby='latent_time', col_color='integrated_snn_res.0.1', n_convolve=100)

In [ ]:
# var_names = [ 'Nanog', 'Dnmt3b']
# scv.pl.scatter(adatamut, var_names, color='integrated_snn_res.0.1', frameon=False)
# scv.pl.scatter(adatamut, x='latent_time', y=var_names, color='integrated_snn_res.0.1', frameon=False)

# Cellrank

In [ ]:
adataWT.obs["integrated_snn_res.0.3"]

In [ ]:
adataWT.uns["initial_states_colors"]=["#F8766D"]

In [ ]:
adatamut.uns["initial_states_colors"]=["#F8766D"]

In [ ]:
adataWT.uns["integrated_snn_res.0.3_colors"]=["#F8766D", "#C49A00", "#53B400", "#00B6EB", "#A58AFF", "#FB61D7"]

In [ ]:
cr.tl.terminal_states(adataWT, cluster_key='integrated_snn_res.0.3', weight_connectivities=0.2)
cr.pl.terminal_states(adataWT,basis='umap',save='terminal_WT_res03.pdf')
cr.tl.terminal_states(adatamut, cluster_key='integrated_snn_res.0.3', weight_connectivities=0.2)
cr.pl.terminal_states(adatamut,basis='umap',save='terminal_mut_res03.pdf')

In [ ]:
cr.tl.initial_states(adataWT, cluster_key='integrated_snn_res.0.3')
cr.pl.initial_states(adataWT, discrete=False,basis='umap',same_plot=False)
cr.pl.initial_states(adataWT, discrete=True,basis='umap',same_plot=False,
                     save='initial_WT_res03.pdf')
cr.tl.initial_states(adatamut, cluster_key='integrated_snn_res.0.3')
cr.pl.initial_states(adatamut, discrete=False,basis='umap',same_plot=False)
cr.pl.initial_states(adatamut, discrete=True,basis='umap',same_plot=False,
                     save='initial_mut_res03.pdf')

In [ ]:
cr.pl.initial_states(adataWT, discrete=True,basis='umap',same_plot=False,dpi=600,
                     save='initial_WT_res03.pdf')

In [ ]:
cr.pl.initial_states(adatamut, discrete=True,basis='umap',same_plot=False,dpi=600,
                     save='initial_mut_res03.pdf')

In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots()
cr.pl.initial_states(adataWT, discrete=True,basis='umap',same_plot=False)



In [ ]:
cr.tl.lineages(adataWT)
cr.tl.lineages(adatamut)

In [ ]:
cr.pl.lineages(adataWT, same_plot=False,basis='umap',save='WT_lineages.pdf')
cr.pl.lineages(adatamut, same_plot=False,basis='umap',save='mut_lineages.pdf')

In [ ]:
scv.tl.recover_latent_time(adataWT, root_key='initial_states_probs', end_key='terminal_states_probs')
cr.tl.lineage_drivers(adataWT)
scv.tl.recover_latent_time(adatamut, root_key='initial_states_probs', end_key='terminal_states_probs')
cr.tl.lineage_drivers(adatamut)

In [ ]:
cr.pl.lineage_drivers(adataWT, lineage="0", n_genes=3,basis='umap')
cr.pl.lineage_drivers(adataWT, lineage="1", n_genes=3,basis='umap')
cr.pl.lineage_drivers(adataWT, lineage="2", n_genes=3,basis='umap')

In [ ]:
cr.pl.lineage_drivers(adatamut, lineage="1", n_genes=5,basis='umap')
cr.pl.lineage_drivers(adatamut, lineage="2", n_genes=5,basis='umap')

In [ ]:
modelWT = cr.ul.models.GAM(adataWT)

labels=['0_corr','1_corr','2_corr']
titles=['0','1','2']

for (lab,tit) in zip(labels,titles):
    top5=adataWT.varm['terminal_lineage_drivers'][lab].sort_values(ascending=False).index[:5]
    cr.pl.gene_trends(adataWT, model=modelWT, data_key='X',
                  genes=top5, ncols=5,
                  time_key='latent_time', same_plot=True, hide_cells=True,
                  figsize=(25, 4), n_test_points=200)

In [ ]:
modelmut = cr.ul.models.GAM(adatamut)

labels=['1_corr','2_corr']
titles=['1','2']

for (lab,tit) in zip(labels,titles):
    top5=adatamut.varm['terminal_lineage_drivers'][lab].sort_values(ascending=False).index[:5]
    cr.pl.gene_trends(adatamut, model=modelmut, data_key='X',
                  genes=top5, ncols=5,
                  time_key='latent_time', same_plot=True, hide_cells=True,
                  figsize=(25, 4), n_test_points=200)

In [ ]:
scv.pl.scatter(adataWT, color='latent_time', color_map='gnuplot', size=80,save='latent_WT.pdf')

In [ ]:
scv.pl.scatter(adatamut, color='latent_time', color_map='gnuplot', size=80,save='latent_mut.pdf')

In [ ]:
cr.pl.heatmap(adataWT, modelWT, genes=adataWT.varm['terminal_lineage_drivers']['0_corr'].sort_values(ascending=False).index[:100],
              show_absorption_probabilities=True,show_all_genes=True,figsize=(10,18),
              lineages="0", n_jobs=1, backend='loky',save='heatmap_lineage_0_WT.pdf')
cr.pl.heatmap(adataWT, modelWT, genes=adataWT.varm['terminal_lineage_drivers']['1_corr'].sort_values(ascending=False).index[:100],
              show_absorption_probabilities=True,show_all_genes=True,figsize=(10,18),
              lineages="1", n_jobs=1, backend='loky',save='heatmap_lineage_1_WT.pdf')
cr.pl.heatmap(adataWT, modelWT, genes=adataWT.varm['terminal_lineage_drivers']['2_corr'].sort_values(ascending=False).index[:100],
              show_absorption_probabilities=True,show_all_genes=True,figsize=(10,18),
              lineages="2", n_jobs=1, backend='loky',save='heatmap_lineage_2_WT.pdf')

In [ ]:
cr.pl.heatmap(adatamut, modelmut, genes=adatamut.varm['terminal_lineage_drivers']['1_corr'].sort_values(ascending=False).index[:100],
              show_absorption_probabilities=True,show_all_genes=True,figsize=(10,18),
              lineages="1", n_jobs=1, backend='loky',save='heatmap_lineage_1_mut.pdf')
cr.pl.heatmap(adatamut, modelmut, genes=adatamut.varm['terminal_lineage_drivers']['2_corr'].sort_values(ascending=False).index[:100],
              show_absorption_probabilities=True,show_all_genes=True,figsize=(10,18),
              lineages="2", n_jobs=1, backend='loky',save='heatmap_lineage_2_mut.pdf')